# Visual Abstract — State-Aware IDP Ensemble Modeling

**9 cells.** Run top-to-bottom. Produces 5 panels + final composite at 300 DPI.

| Row | Panel | Content |
|-----|-------|---------|
| Problem | A | IDP Ensemble Reality |
| Problem | B | Static Structure Failure |
| Method | C | Pocket-State Modeling (tICA/PCA scatter) |
| Method | D | SE(3)-Equivariant Diffusion Schematic |
| Outcome | E | Validation Funnel |

Two modes:
- **With real data**: upload your topology.pdb + trajectory.xtc when prompted.
- **Demo mode**: skip the upload — synthetic IDP ensemble is generated automatically.

In [ ]:
# ╔══════════════════════════════════════════════════════════════════╗
# ║  CELL 1 — INSTALL DEPENDENCIES                                 ║
# ╚══════════════════════════════════════════════════════════════════╝

!pip install -q --upgrade pip
!pip install -q numpy scipy matplotlib seaborn Pillow
!pip install -q mdtraj scikit-learn pandas
print('\n--- All packages installed ---')

In [ ]:
# ╔══════════════════════════════════════════════════════════════════╗
# ║  CELL 2 — IMPORTS, PROJECT LAYOUT & PUBLICATION STYLE          ║
# ╚══════════════════════════════════════════════════════════════════╝

import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.patches import FancyBboxPatch, Ellipse, Polygon, Circle
from matplotlib.colors import LinearSegmentedColormap, Normalize
from matplotlib.collections import LineCollection
from matplotlib import cm, patheffects
from PIL import Image, ImageDraw, ImageFont
import seaborn as sns
from scipy.ndimage import gaussian_filter
from pathlib import Path
import json, os, datetime, platform, subprocess, sys, warnings, shutil
warnings.filterwarnings('ignore')

# ── Publication rcParams (Nature / Science style) ──
plt.rcParams.update({
    'font.family': 'sans-serif',
    'font.sans-serif': ['Arial', 'Helvetica', 'DejaVu Sans'],
    'font.size': 8,
    'axes.titlesize': 10,
    'axes.labelsize': 9,
    'xtick.labelsize': 7,
    'ytick.labelsize': 7,
    'legend.fontsize': 7,
    'figure.dpi': 150,
    'savefig.dpi': 300,
    'axes.linewidth': 0.6,
    'xtick.major.width': 0.5,
    'ytick.major.width': 0.5,
    'lines.linewidth': 1.0,
    'axes.spines.top': False,
    'axes.spines.right': False,
    'pdf.fonttype': 42,
    'ps.fonttype': 42,
})

# ── Colorblind-safe palette (Wong 2011, Nature Methods) ──
PAL = {
    'bg':           '#FFFFFF',
    'text':         '#1B1B1B',
    'open':         '#E69F00',   # orange  — OPEN pocket
    'closed':       '#0072B2',   # blue    — CLOSED pocket
    'highlight':    '#D55E00',   # vermilion — emphasis
    'pocket':       '#F0E442',   # yellow  — transient pocket
    'generated':    '#CC79A7',   # reddish-purple — diffusion output
    'model_box':    '#7C3AED',   # purple — model
    'model_bg':     '#EDE9FE',
    'model_dark':   '#4C1D95',
    'grey':         '#999999',
    'light_grey':   '#E5E5E5',
    'funnel': [
        '#56B4E9',  # sky blue — stage 1
        '#E69F00',  # orange   — stage 2
        '#CC79A7',  # reddish-purple — stage 3
        '#009E73',  # bluish-green — stage 4
    ],
}

DPI = 300
# Nature single-column = 89 mm ≈ 3.5 in; double = 183 mm ≈ 7.2 in
COL1 = 3.5
COL2 = 7.2

# ── Project layout ──
ROOT = Path('visual_abstract')
PANELS = ROOT / 'panels'
EXPORTS = ROOT / 'exports'
DATA = ROOT / 'data'
for d in [PANELS, EXPORTS, DATA]:
    d.mkdir(parents=True, exist_ok=True)

# ── Provenance ──
meta = {
    'created': datetime.datetime.utcnow().isoformat() + 'Z',
    'python': sys.version.split()[0],
    'numpy': np.__version__,
    'matplotlib': matplotlib.__version__,
}
(ROOT / 'provenance.json').write_text(json.dumps(meta, indent=2))
print('Project root:', ROOT)
print('Palette preview:'); [print(f'  {k}: {v}') for k, v in PAL.items() if isinstance(v, str)];

In [ ]:
# ╔══════════════════════════════════════════════════════════════════╗
# ║  CELL 3 — DATA: UPLOAD REAL TRAJECTORY  *OR*  GENERATE DEMO    ║
# ╚══════════════════════════════════════════════════════════════════╝
#
# Option A: Upload topology.pdb + trajectory.xtc  (uncomment block A)
# Option B: Use synthetic IDP ensemble for demo    (default)

USE_REAL_DATA = False  # <-- set True if uploading real MD data

if USE_REAL_DATA:
    # ── BLOCK A: Real data upload ──
    from google.colab import files as colab_files
    import mdtraj as md

    print('Upload your topology (.pdb/.gro) and trajectory (.xtc/.dcd):')
    uploaded = colab_files.upload()
    for fn in uploaded:
        shutil.move(f'/content/{fn}', str(DATA / fn))

    TOPOLOGY  = DATA / 'topology.pdb'       # ← edit filename
    TRAJECTORY = DATA / 'trajectory.xtc'     # ← edit filename

    traj = md.load(str(TRAJECTORY), top=str(TOPOLOGY))
    stride = max(1, traj.n_frames // 5000)
    traj_ds = traj[::stride]
    print(f'Loaded {traj.n_frames} frames → downsampled to {traj_ds.n_frames}')

    # Extract CA coordinates as ensemble
    ca = traj_ds.topology.select('name CA')
    ensemble = [traj_ds.xyz[i, ca, :] * 10.0 for i in range(traj_ds.n_frames)]  # nm→Å
    # Center each conformer
    ensemble = [c - c.mean(axis=0) for c in ensemble]
    # Subsample for visualization
    vis_idx = np.linspace(0, len(ensemble)-1, min(40, len(ensemble)), dtype=int)
    ensemble_vis = [ensemble[i] for i in vis_idx]

    # Also export PDBs for external rendering
    pdb_dir = EXPORTS / 'ensemble_pdbs'
    pdb_dir.mkdir(exist_ok=True)
    for i, fi in enumerate(vis_idx):
        traj_ds[fi].save_pdb(str(pdb_dir / f'conf_{i:03d}.pdb'))
    print(f'Exported {len(vis_idx)} PDBs to {pdb_dir}')

else:
    # ── BLOCK B: Synthetic demo ensemble ──
    def _generate_idp_ensemble(n_conf=40, n_res=80, seed=42):
        """Persistent random walk on a sphere → IDP-like backbone."""
        rng = np.random.default_rng(seed)
        out = []
        for _ in range(n_conf):
            coords = np.zeros((n_res, 3))
            d = rng.standard_normal(3)
            d /= np.linalg.norm(d)
            for j in range(1, n_res):
                step = d + 0.55 * rng.standard_normal(3)
                step = 3.8 * step / np.linalg.norm(step)
                coords[j] = coords[j-1] + step
                d = step / np.linalg.norm(step)
            coords -= coords.mean(axis=0)
            out.append(coords)
        return out

    ensemble_vis = _generate_idp_ensemble(40, 80)
    ensemble = ensemble_vis  # same object in demo mode
    print(f'Generated synthetic ensemble: {len(ensemble_vis)} conformers, '
          f'{ensemble_vis[0].shape[0]} residues')
    print('(Set USE_REAL_DATA=True in this cell to use your own trajectory)')

In [ ]:
# ╔══════════════════════════════════════════════════════════════════╗
# ║  CELL 4 — PANEL 1: IDP ENSEMBLE REALITY                       ║
# ║  "IDPs are flexible ensembles, not single folds."              ║
# ╚══════════════════════════════════════════════════════════════════╝
#
# 40 semi-transparent backbone traces overlaid, colored N→C (blue→red)
# with a smooth density halo → "fuzzy cloud" effect.

fig = plt.figure(figsize=(COL1 + 1, COL1 + 0.5), facecolor='white')
ax = fig.add_subplot(111, projection='3d', facecolor='white')

cmap_nc = LinearSegmentedColormap.from_list(
    'NtoC', [PAL['closed'], '#FFFFFF', PAL['highlight']])
n_res = ensemble_vis[0].shape[0]
t_norm = np.linspace(0, 1, n_res)

for ci, conf in enumerate(ensemble_vis):
    # Smooth backbone for cleaner lines
    t_fine = np.linspace(0, 1, 250)
    smooth = np.column_stack(
        [np.interp(t_fine, t_norm, conf[:, k]) for k in range(3)])
    colors = cmap_nc(t_fine)
    colors[:, 3] = 0.18  # semi-transparent
    for i in range(len(smooth) - 1):
        ax.plot(smooth[i:i+2, 0], smooth[i:i+2, 1], smooth[i:i+2, 2],
                color=colors[i], linewidth=0.9, solid_capstyle='round')

# ── Labels ──
ax.text2D(0.5, 0.97, 'BIOLOGICAL PROBLEM', transform=ax.transAxes,
          ha='center', fontsize=7, color=PAL['grey'], fontweight='bold',
          fontstyle='italic')
ax.set_title('IDP Ensemble Reality', fontsize=11, fontweight='bold',
             color=PAL['text'], pad=22)
ax.text2D(0.5, 0.01,
          'IDPs exist as flexible conformational ensembles,\n'
          'not a single static fold.',
          transform=ax.transAxes, ha='center', fontsize=7.5,
          color=PAL['text'], style='italic',
          bbox=dict(boxstyle='round,pad=0.3', fc='#F5F5F5', ec='none', alpha=0.8))

# ── Colorbar (N→C terminus) ──
sm = plt.cm.ScalarMappable(cmap=cmap_nc, norm=Normalize(0, 1))
cbar = fig.colorbar(sm, ax=ax, shrink=0.35, aspect=12, pad=0.01)
cbar.set_ticks([0, 1])
cbar.set_ticklabels(['N-term', 'C-term'])
cbar.ax.tick_params(labelsize=6)

ax.set_axis_off()
ax.view_init(elev=18, azim=42)
plt.tight_layout()
plt.savefig(PANELS / 'panel1_idp_ensemble.png', dpi=DPI, bbox_inches='tight',
            facecolor='white')
plt.savefig(PANELS / 'panel1_idp_ensemble.svg', bbox_inches='tight',
            facecolor='white')
plt.show()
print('Panel 1 saved (PNG + SVG)')

In [ ]:
# ╔══════════════════════════════════════════════════════════════════╗
# ║  CELL 5 — PANEL 2: STATIC STRUCTURE FAILURE                   ║
# ║  "Static docking misses transient cryptic pockets."            ║
# ╚══════════════════════════════════════════════════════════════════╝
#
# Left: one rigid backbone (high opacity).  Right: full ensemble
# cloud with 3 transient pocket spheres highlighted in yellow.

fig = plt.figure(figsize=(COL2, COL1 + 0.3), facecolor='white')

# ── Left: Single static structure ──
ax1 = fig.add_subplot(121, projection='3d', facecolor='white')
single = ensemble_vis[0]
t_fine = np.linspace(0, 1, 300)
smooth = np.column_stack(
    [np.interp(t_fine, np.linspace(0, 1, n_res), single[:, k]) for k in range(3)])
ax1.plot(smooth[:, 0], smooth[:, 1], smooth[:, 2],
         color=PAL['closed'], linewidth=2.2, alpha=0.92, solid_capstyle='round')
ax1.set_title('Static docking target', fontsize=9, fontweight='bold',
              color=PAL['text'], pad=10)
# ✕ marker indicating failure
ax1.text2D(0.85, 0.85, '\u2717', transform=ax1.transAxes,
           fontsize=22, color=PAL['highlight'], fontweight='bold', ha='center')
ax1.set_axis_off()
ax1.view_init(elev=18, azim=42)

# ── Right: Ensemble + transient pockets ──
ax2 = fig.add_subplot(122, projection='3d', facecolor='white')
cmap_nc = LinearSegmentedColormap.from_list(
    'NtoC', [PAL['closed'], '#FFFFFF', PAL['highlight']])

for conf in ensemble_vis:
    t_fine = np.linspace(0, 1, 250)
    sm_c = np.column_stack(
        [np.interp(t_fine, np.linspace(0, 1, n_res), conf[:, k]) for k in range(3)])
    colors = cmap_nc(t_fine)
    colors[:, 3] = 0.12
    for i in range(len(sm_c) - 1):
        ax2.plot(sm_c[i:i+2, 0], sm_c[i:i+2, 1], sm_c[i:i+2, 2],
                 color=colors[i], linewidth=0.6)

# Transient pocket spheres
rng = np.random.default_rng(99)
pocket_frames = [3, 12, 25]
pocket_residues = [rng.integers(20, 60) for _ in pocket_frames]
u_s, v_s = np.mgrid[0:2*np.pi:18j, 0:np.pi:9j]
for fi, ri in zip(pocket_frames, pocket_residues):
    pc = ensemble_vis[fi][ri]
    r = 5.5
    ax2.plot_surface(
        pc[0] + r * np.cos(u_s) * np.sin(v_s),
        pc[1] + r * np.sin(u_s) * np.sin(v_s),
        pc[2] + r * np.cos(v_s),
        color=PAL['pocket'], alpha=0.30, edgecolor='none')

ax2.set_title('Dynamic ensemble +\ntransient cryptic pockets',
              fontsize=9, fontweight='bold', color=PAL['text'], pad=10)
# ✓ marker
ax2.text2D(0.85, 0.85, '\u2713', transform=ax2.transAxes,
           fontsize=22, color=PAL['open'], fontweight='bold', ha='center')
ax2.set_axis_off()
ax2.view_init(elev=18, azim=42)

# ── Divider + caption ──
fig.text(0.5, 1.01, 'STATIC STRUCTURE FAILURE', ha='center',
         fontsize=11, fontweight='bold', color=PAL['text'])
fig.text(0.5, -0.03,
         'Static docking on a single PDB misses transient, cryptic binding pockets '
         'that appear only in the dynamic ensemble.',
         ha='center', fontsize=8, color=PAL['text'], style='italic',
         bbox=dict(boxstyle='round,pad=0.3', fc='#F5F5F5', ec='none', alpha=0.8))

plt.tight_layout(w_pad=1.5)
plt.savefig(PANELS / 'panel2_static_failure.png', dpi=DPI, bbox_inches='tight',
            facecolor='white')
plt.savefig(PANELS / 'panel2_static_failure.svg', bbox_inches='tight',
            facecolor='white')
plt.show()
print('Panel 2 saved (PNG + SVG)')

In [ ]:
# ╔══════════════════════════════════════════════════════════════════╗
# ║  CELL 6 — PANEL 3: POCKET-STATE MODELING (LANDSCAPE)          ║
# ║  "Cluster ensemble into OPEN vs CLOSED macrostates."           ║
# ╚══════════════════════════════════════════════════════════════════╝
#
# Left: 2D scatter (tIC1 vs tIC2) colored by macrostate with KDE contours.
# Right: Representative conformers for OPEN and CLOSED states.

from sklearn.decomposition import PCA
from sklearn.cluster import KMeans

# ── Feature extraction ──
# If using real data, we compute pairwise CA distances.
# For synthetic data, we use flattened coordinates directly.
if USE_REAL_DATA:
    import mdtraj as md
    ca_idx = traj_ds.topology.select('name CA')
    rng_f = np.random.default_rng(7)
    n_pairs = min(300, len(ca_idx) * (len(ca_idx) - 1) // 2)
    pairs, seen = [], set()
    while len(pairs) < n_pairs:
        a, b = sorted(rng_f.choice(ca_idx, size=2, replace=False))
        if (a, b) not in seen:
            seen.add((a, b))
            pairs.append([a, b])
    D_feat = md.compute_distances(traj_ds, np.array(pairs))
else:
    # Flatten coordinates as features for synthetic data
    D_feat = np.array([c.flatten() for c in ensemble])

# ── 2D projection ──
pca = PCA(n_components=2, random_state=7)
X2 = pca.fit_transform(D_feat)
var_exp = pca.explained_variance_ratio_ * 100

# ── Clustering + macrostate assignment ──
km = KMeans(n_clusters=6, random_state=7, n_init=20)
cluster_labels = km.fit_predict(X2)

# Openness proxy: distance from global centroid (higher = more extended = OPEN)
centroid = D_feat.mean(axis=0, keepdims=True)
openness = np.sqrt(((D_feat - centroid) ** 2).sum(axis=1))
macro = np.where(openness >= np.median(openness), 'OPEN', 'CLOSED')

df_map = pd.DataFrame({
    'x': X2[:, 0], 'y': X2[:, 1],
    'cluster': cluster_labels, 'macro': macro, 'openness': openness})
df_map.to_csv(DATA / 'state_map_points.csv', index=False)

# ── Representative frames (medoid per macrostate) ──
def _medoid(indices):
    pts = X2[indices]
    c = pts.mean(axis=0, keepdims=True)
    return indices[np.argmin(((pts - c)**2).sum(axis=1))]

idx_open = np.where(macro == 'OPEN')[0]
idx_closed = np.where(macro == 'CLOSED')[0]
rep_open = _medoid(idx_open)
rep_closed = _medoid(idx_closed)

# ── Figure ──
fig = plt.figure(figsize=(COL2, COL1), facecolor='white')
gs = fig.add_gridspec(1, 5, wspace=0.05)

# Left 3/5: scatter
ax_sc = fig.add_subplot(gs[0, :3])
ax_sc.set_facecolor('#FAFAFA')

# KDE contours
for label, color in [('CLOSED', PAL['closed']), ('OPEN', PAL['open'])]:
    mask = macro == label
    if mask.sum() > 10:
        try:
            sns.kdeplot(x=X2[mask, 0], y=X2[mask, 1], ax=ax_sc,
                        levels=3, color=color, linewidths=0.7, alpha=0.5)
        except Exception:
            pass  # skip if too few points for KDE

# Scatter points
for label, color, marker in [('CLOSED', PAL['closed'], 'o'), ('OPEN', PAL['open'], 's')]:
    mask = macro == label
    ax_sc.scatter(X2[mask, 0], X2[mask, 1], c=color, s=14, alpha=0.6,
                  edgecolors='none', label=label, marker=marker, zorder=3)

# Highlight representatives
for ri, color, lbl in [(rep_open, PAL['open'], 'OPEN rep'),
                        (rep_closed, PAL['closed'], 'CLOSED rep')]:
    ax_sc.scatter(X2[ri, 0], X2[ri, 1], s=90, facecolors='none',
                  edgecolors=color, linewidths=1.8, zorder=5)

ax_sc.set_xlabel(f'Component 1 ({var_exp[0]:.1f}% var.)', fontsize=8)
ax_sc.set_ylabel(f'Component 2 ({var_exp[1]:.1f}% var.)', fontsize=8)
ax_sc.legend(fontsize=7, frameon=False, loc='upper right')
ax_sc.set_title('Conformational landscape', fontsize=10, fontweight='bold')

# Right 2/5: representative structures
ax_top = fig.add_subplot(gs[0, 3], projection='3d', facecolor='white')
ax_bot = fig.add_subplot(gs[0, 4], projection='3d', facecolor='white')

for ax_r, ri, color, title in [
    (ax_top, rep_open, PAL['open'], 'OPEN'),
    (ax_bot, rep_closed, PAL['closed'], 'CLOSED')]:
    conf = ensemble_vis[min(ri, len(ensemble_vis)-1)]
    t_f = np.linspace(0, 1, 200)
    sm_r = np.column_stack(
        [np.interp(t_f, np.linspace(0, 1, conf.shape[0]), conf[:, k]) for k in range(3)])
    ax_r.plot(sm_r[:, 0], sm_r[:, 1], sm_r[:, 2],
              color=color, linewidth=1.8, alpha=0.85, solid_capstyle='round')
    if title == 'OPEN':
        # pocket highlight
        mid = conf[conf.shape[0]//2]
        u_p, v_p = np.mgrid[0:2*np.pi:12j, 0:np.pi:6j]
        rp = 4.5
        ax_r.plot_surface(
            mid[0]+rp*np.cos(u_p)*np.sin(v_p),
            mid[1]+rp*np.sin(u_p)*np.sin(v_p),
            mid[2]+rp*np.cos(v_p),
            color=PAL['pocket'], alpha=0.3, edgecolor='none')
    ax_r.set_title(f'{title} state', fontsize=8, fontweight='bold', color=color, pad=2)
    ax_r.set_axis_off()
    ax_r.view_init(elev=18, azim=42)

fig.text(0.5, -0.04,
         'Ensemble clustered into pocket-centric macrostates (OPEN vs CLOSED).',
         ha='center', fontsize=8, color=PAL['text'], style='italic',
         bbox=dict(boxstyle='round,pad=0.3', fc='#F5F5F5', ec='none', alpha=0.8))

plt.savefig(PANELS / 'panel3_state_map.png', dpi=DPI, bbox_inches='tight',
            facecolor='white')
plt.savefig(PANELS / 'panel3_state_map.svg', bbox_inches='tight',
            facecolor='white')
plt.show()
print(f'Panel 3 saved. OPEN rep: frame {rep_open}, CLOSED rep: frame {rep_closed}')

In [ ]:
# ╔══════════════════════════════════════════════════════════════════╗
# ║  CELL 7 — PANEL 4: SE(3)-EQUIVARIANT DIFFUSION SCHEMATIC      ║
# ║  Three-stage flow: Noisy → Diffusion model → OPEN ensemble     ║
# ╚══════════════════════════════════════════════════════════════════╝

fig, ax = plt.subplots(figsize=(COL2, 2.0), facecolor='white')
ax.set_xlim(0, 16)
ax.set_ylim(-0.2, 5.2)
ax.set_axis_off()
ax.set_aspect('equal')

rng_d = np.random.default_rng(42)

# ══════ STAGE 1: Noisy structures ══════
for i in range(4):
    x0 = 0.4 + i * 0.85
    y0 = 2.0 + rng_d.uniform(-0.4, 0.4)
    xs = x0 + np.cumsum(rng_d.uniform(0.04, 0.14, 14))
    ys = y0 + np.cumsum(rng_d.normal(0, 0.2, 14))
    ax.plot(xs, ys, color=PAL['grey'], linewidth=1.2, alpha=0.55,
            solid_capstyle='round')
    # Gaussian noise dots
    for _ in range(10):
        ax.plot(rng_d.choice(xs) + rng_d.normal(0, 0.1),
                rng_d.choice(ys) + rng_d.normal(0, 0.1),
                '.', color=PAL['grey'], markersize=1.5, alpha=0.25)

# Label box
ax.add_patch(FancyBboxPatch((0.1, 0.15), 3.9, 0.65,
    boxstyle='round,pad=0.12', fc=PAL['light_grey'], ec=PAL['grey'], lw=0.8))
ax.text(2.05, 0.47, 'Noisy structures (t)', ha='center', va='center',
        fontsize=8, fontweight='bold', color=PAL['text'])

# Time indicator
ax.annotate('t = T', xy=(0.6, 4.2), fontsize=7, color=PAL['grey'],
            fontweight='bold')

# ══════ ARROW 1 ══════
ax.annotate('', xy=(5.3, 2.5), xytext=(4.3, 2.5),
            arrowprops=dict(arrowstyle='->', color=PAL['text'],
                            lw=1.8, connectionstyle='arc3,rad=0'))
ax.text(4.8, 2.9, 'denoise', fontsize=6.5, ha='center', color=PAL['grey'],
        fontstyle='italic')

# ══════ STAGE 2: SE(3) model box ══════
ax.add_patch(FancyBboxPatch((5.3, 0.6), 5.4, 3.8,
    boxstyle='round,pad=0.25', fc=PAL['model_bg'], ec=PAL['model_box'], lw=1.8))
ax.text(8.0, 3.8, 'SE(3)-Equivariant\nDiffusion Model',
        ha='center', va='center', fontsize=9.5, fontweight='bold',
        color=PAL['model_dark'])

# Rotation symbol (circular arrow)
ang = np.linspace(0, 1.65 * np.pi, 60)
r_c = 0.55
cx, cy = 8.0, 2.0
ax.plot(cx + r_c * np.cos(ang), cy + r_c * np.sin(ang),
        color=PAL['model_box'], linewidth=1.5, alpha=0.7)
ax.annotate('', xy=(cx + r_c * np.cos(ang[-1]), cy + r_c * np.sin(ang[-1])),
            xytext=(cx + r_c * np.cos(ang[-4]), cy + r_c * np.sin(ang[-4])),
            arrowprops=dict(arrowstyle='->', color=PAL['model_box'], lw=1.5))

# Translation symbol
ax.annotate('', xy=(7.0, 2.0), xytext=(6.4, 2.0),
            arrowprops=dict(arrowstyle='->', color=PAL['model_box'], lw=1.2))
ax.text(6.7, 2.2, 'T', fontsize=7, color=PAL['model_box'], fontweight='bold')
ax.text(8.5, 1.35, 'R', fontsize=7, color=PAL['model_box'], fontweight='bold')

# Mini backbone in center
mx = np.array([7.4, 7.65, 7.9, 8.15, 8.4, 8.6])
my = np.array([2.0, 2.2, 1.85, 2.1, 1.8, 2.0])
ax.plot(mx, my, color=PAL['model_box'], linewidth=2.0, alpha=0.7,
        solid_capstyle='round')

# Conditioning label
ax.add_patch(FancyBboxPatch((5.6, -0.05), 4.8, 0.5,
    boxstyle='round,pad=0.08', fc=PAL['open'], ec=PAL['open'],
    alpha=0.15, lw=1.0))
ax.text(8.0, 0.2, 'conditioned on OPEN macrostate', ha='center', va='center',
        fontsize=7, fontweight='bold', color='#7C4A00')

# ══════ ARROW 2 ══════
ax.annotate('', xy=(12.0, 2.5), xytext=(11.0, 2.5),
            arrowprops=dict(arrowstyle='->', color=PAL['text'],
                            lw=1.8, connectionstyle='arc3,rad=0'))
ax.text(11.5, 2.9, 't \u2192 0', fontsize=6.5, ha='center', color=PAL['grey'],
        fontstyle='italic')

# ══════ STAGE 3: Generated OPEN-state ensemble ══════
for i in range(4):
    x0 = 12.2 + i * 0.5
    y0 = 1.6 + rng_d.uniform(-0.15, 0.15)
    xs = x0 + np.cumsum(rng_d.uniform(0.04, 0.1, 16))
    ys = y0 + np.cumsum(rng_d.normal(0, 0.06, 16))
    ax.plot(xs, ys, color=PAL['generated'], linewidth=1.6, alpha=0.7,
            solid_capstyle='round')

# Pocket highlight
ax.add_patch(Circle((14.0, 2.3), 0.45, fc=PAL['pocket'], alpha=0.25,
                     ec=PAL['open'], lw=1.2, ls='--'))

# Label box
ax.add_patch(FancyBboxPatch((12.0, 0.15), 3.8, 0.65,
    boxstyle='round,pad=0.12', fc='#FCE7F3', ec=PAL['generated'], lw=0.8))
ax.text(13.9, 0.47, 'Generated OPEN-state\nensemble', ha='center', va='center',
        fontsize=7.5, fontweight='bold', color=PAL['text'])

ax.annotate('t = 0', xy=(14.5, 4.2), fontsize=7, color=PAL['grey'],
            fontweight='bold')

ax.set_title('SE(3)-EQUIVARIANT DIFFUSION', fontsize=11,
             fontweight='bold', color=PAL['text'], pad=10)
fig.text(0.5, -0.08,
         'State-aware SE(3) diffusion generates 3D structures '
         'conditioned on the target pocket macrostate.',
         ha='center', fontsize=8, color=PAL['text'], style='italic',
         bbox=dict(boxstyle='round,pad=0.3', fc='#F5F5F5', ec='none', alpha=0.8))

plt.savefig(PANELS / 'panel4_diffusion.png', dpi=DPI, bbox_inches='tight',
            facecolor='white')
plt.savefig(PANELS / 'panel4_diffusion.svg', bbox_inches='tight',
            facecolor='white')
plt.show()
print('Panel 4 saved (PNG + SVG)')

In [ ]:
# ╔══════════════════════════════════════════════════════════════════╗
# ║  CELL 8 — PANEL 5: VALIDATION FUNNEL                          ║
# ║  4-stage funnel: generation → poses → selectivity → binders    ║
# ╚══════════════════════════════════════════════════════════════════╝
#
# Replace the `counts` array with your actual screening numbers.

stages = [
    'Generated\nmolecules',
    'Valid OPEN-state\nposes',
    'Selective \u0394\u0394G\n(open > closed)',
    'State-selective\nbinders',
]
counts = np.array([120_000, 8_500, 420, 19])  # ← edit with real numbers
colors_f = PAL['funnel']

fig, ax = plt.subplots(figsize=(COL1 + 0.8, COL2 - 0.5), facecolor='white')
ax.set_xlim(-5.5, 5.5)
ax.set_ylim(-0.5, 10.5)
ax.set_axis_off()
ax.set_aspect('equal')

y_tops = [9.0, 6.7, 4.4, 2.1]
h = 2.0
w_tops = [4.5, 3.6, 2.6, 1.6]
w_bots = [3.6, 2.6, 1.6, 0.8]

rng_f = np.random.default_rng(10)

for i, (stage, yt, wt, wb) in enumerate(zip(stages, y_tops, w_tops, w_bots)):
    yb = yt - h
    trap = Polygon(
        [[-wt, yt], [wt, yt], [wb, yb], [-wb, yb]],
        closed=True, fc=colors_f[i], ec='white', lw=2.0, alpha=0.88)
    ax.add_patch(trap)

    # Stage label + count
    ax.text(0, (yt + yb) / 2 + 0.15, stage, ha='center', va='center',
            fontsize=8, fontweight='bold', color='white',
            path_effects=[patheffects.withStroke(linewidth=2, foreground='#333')])
    ax.text(0, (yt + yb) / 2 - 0.55, f'N = {counts[i]:,}',
            ha='center', va='center', fontsize=7, color='white',
            fontstyle='italic',
            path_effects=[patheffects.withStroke(linewidth=1.5, foreground='#333')])

    # Molecule icons (small circles)
    n_icons = max(2, int(8 * (1 - i / len(stages))))
    for _ in range(n_icons):
        ix = rng_f.uniform(-wt * 0.4, wt * 0.4)
        iy = rng_f.uniform(yb + 0.3, yt - 0.3)
        ax.plot(ix, iy, 'o', color='white', markersize=2 + i,
                alpha=0.4, markeredgewidth=0)

# Side arrow
ax.annotate('', xy=(-5.0, 0.8), xytext=(-5.0, 10.0),
            arrowprops=dict(arrowstyle='->', color=PAL['text'], lw=2.0))
ax.text(-5.0, 5.5, 'In silico +\nexperimental\nfiltering',
        ha='center', va='center', fontsize=7.5, fontweight='bold',
        color=PAL['text'], rotation=90)

# Attrition percentages on right
for i in range(len(counts) - 1):
    pct = counts[i+1] / counts[i] * 100
    y_mid = (y_tops[i] - h + y_tops[i+1]) / 2
    ax.text(w_tops[i] + 0.6, y_mid, f'{pct:.1f}%\npass',
            ha='left', va='center', fontsize=6.5, color=PAL['grey'],
            fontstyle='italic')

ax.set_title('VALIDATION FUNNEL', fontsize=11, fontweight='bold',
             color=PAL['text'], pad=12)
fig.text(0.5, 0.0,
         'Multistage filtering yields experimentally verified,\n'
         'state-selective binders that stabilize the OPEN pocket.',
         ha='center', fontsize=8, color=PAL['text'], style='italic',
         bbox=dict(boxstyle='round,pad=0.3', fc='#F5F5F5', ec='none', alpha=0.8))

plt.savefig(PANELS / 'panel5_funnel.png', dpi=DPI, bbox_inches='tight',
            facecolor='white')
plt.savefig(PANELS / 'panel5_funnel.svg', bbox_inches='tight',
            facecolor='white')
plt.show()
print('Panel 5 saved (PNG + SVG)')

In [ ]:
# ╔══════════════════════════════════════════════════════════════════╗
# ║  CELL 9 — FINAL COMPOSITE + DOWNLOAD                          ║
# ║  Assembles all 5 panels into one figure with row labels.       ║
# ╚══════════════════════════════════════════════════════════════════╝

from PIL import Image, ImageDraw, ImageFont
import glob, zipfile

# ── Load panels ──
names = [
    'panel1_idp_ensemble.png',
    'panel2_static_failure.png',
    'panel3_state_map.png',
    'panel4_diffusion.png',
    'panel5_funnel.png',
]
imgs = {n: Image.open(PANELS / n) for n in names}
print('Loaded:', list(imgs.keys()))

# ── Layout constants ──
TARGET_W = 4200  # pixels (~183 mm at 600 dpi = journal double-column)
MARGIN = 50
COL_W = (TARGET_W - 3 * MARGIN) // 2

def _resize(img, w):
    h = int(img.height * w / img.width)
    return img.resize((w, h), Image.LANCZOS)

p1 = _resize(imgs[names[0]], COL_W)
p2 = _resize(imgs[names[1]], COL_W)
p3 = _resize(imgs[names[2]], COL_W)
p4 = _resize(imgs[names[3]], COL_W)
p5 = _resize(imgs[names[4]], TARGET_W - 2 * MARGIN)

ROW1_H = max(p1.height, p2.height)
ROW2_H = max(p3.height, p4.height)
ROW3_H = p5.height
TITLE_H = 110
ROW_LABEL_H = 30
TOTAL_H = TITLE_H + 3 * ROW_LABEL_H + 4 * MARGIN + ROW1_H + ROW2_H + ROW3_H

comp = Image.new('RGB', (TARGET_W, TOTAL_H), 'white')
draw = ImageDraw.Draw(comp)

# Fonts
try:
    font_title = ImageFont.truetype('/usr/share/fonts/truetype/dejavu/DejaVuSans-Bold.ttf', 48)
    font_label = ImageFont.truetype('/usr/share/fonts/truetype/dejavu/DejaVuSans-Bold.ttf', 34)
    font_row   = ImageFont.truetype('/usr/share/fonts/truetype/dejavu/DejaVuSans.ttf', 24)
except Exception:
    font_title = font_label = font_row = ImageFont.load_default()

# Title
draw.text((TARGET_W // 2, 30),
          'Visual Abstract: State-Aware IDP Ensemble Modeling',
          fill='#1B1B1B', font=font_title, anchor='mt')

y = TITLE_H
row_data = [
    ('PROBLEM', p1, p2, ROW1_H),
    ('METHOD',  p3, p4, ROW2_H),
]
panel_letters = iter('ABCDE')

for row_label, pa, pb, rh in row_data:
    # Row label
    draw.text((MARGIN, y + 2), row_label, fill='#888888', font=font_row)
    y += ROW_LABEL_H
    # Panels
    comp.paste(pa, (MARGIN, y))
    comp.paste(pb, (MARGIN + COL_W + MARGIN, y))
    # Panel letters
    la, lb = next(panel_letters), next(panel_letters)
    draw.text((MARGIN + 8, y + 5), la, fill='#1B1B1B', font=font_label)
    draw.text((MARGIN + COL_W + MARGIN + 8, y + 5), lb, fill='#1B1B1B', font=font_label)
    y += rh + MARGIN

# Bottom row (spanning)
draw.text((MARGIN, y + 2), 'OUTCOME', fill='#888888', font=font_row)
y += ROW_LABEL_H
comp.paste(p5, (MARGIN, y))
draw.text((MARGIN + 8, y + 5), next(panel_letters), fill='#1B1B1B', font=font_label)

comp.save(str(ROOT / 'visual_abstract_composite.png'), dpi=(300, 300))
comp.save(str(ROOT / 'visual_abstract_composite.tiff'),
          compression='tiff_lzw', dpi=(300, 300))

print(f'\nComposite: {comp.size[0]} x {comp.size[1]} px')

# ── ZIP everything ──
zip_path = ROOT / 'visual_abstract_all.zip'
with zipfile.ZipFile(zip_path, 'w', zipfile.ZIP_DEFLATED) as zf:
    for f in sorted(PANELS.glob('*')):
        zf.write(f, f'panels/{f.name}')
    for ext in ['png', 'tiff']:
        p = ROOT / f'visual_abstract_composite.{ext}'
        if p.exists():
            zf.write(p, p.name)
    zf.write(ROOT / 'provenance.json', 'provenance.json')
    if (DATA / 'state_map_points.csv').exists():
        zf.write(DATA / 'state_map_points.csv', 'data/state_map_points.csv')

print(f'ZIP: {zip_path}')

# ── Download ──
try:
    from google.colab import files as colab_files
    colab_files.download(str(zip_path))
except ImportError:
    print('Not in Colab — zip is at:', zip_path)

# ── Preview ──
display(comp.resize((comp.width // 3, comp.height // 3), Image.LANCZOS))